# Agregação Hierárquica de Tópicos: Condensando Micro-Tópicos do LLDA com LDA Clássico

**Experimento de Origem:** `experimento_comparativo_aapd.py`  
**Dataset Base:** ArXiv Academic Paper Dataset (AAPD) — 2.000 artigos científicos  
**Arquivo de Entrada:** `experimento_aapd_comparativo.csv`

---

### Motivação e Abordagem em Dois Estágios
1. **Estágio 1 (Extração de Micro-Tópicos pelo LLDA):** O modelo generativo (LLM via Ollama) formulou títulos conceituais de alta especificidade para cada resumo acadêmico. Como resultado, foram gerados **1.989 tópicos distintos em 2.000 documentos** (Distinct-2 de 84,01%).
2. **Estágio 2 (Agregação em Macro-Tópicos via LDA):** Embora a alta especificidade do LLDA seja excelente para detalhamento individual, é impraticável para taxonomia e sumarização global. Neste notebook, utilizamos o **Latent Dirichlet Allocation (LDA)** probabilístico como um mecanismo de **agregação não supervisionada** para condensar os quase 2.000 micro-tópicos em um conjunto enxuto, coerente e balanceado de **Macro-Tópicos** ($K=10$).

---

### Roteiro da Análise:
* **Vetorização Vocabular:** Engenharia de termos e remoção de meta-palavras acadêmicas genéricas.
* **Busca de Hiperparâmetros:** Análise de perplexidade para seleção do número ideal de macro-tópicos ($K$).
* **Modelagem e Rotulagem:** Ajuste do LDA e síntese de descrições dos clusters temáticos.
* **Inspeção Qualitativa:** Mapeamento demonstrando como micro-tópicos específicos foram agrupados.
* **Validação Cruzada:** Heatmap de coerência com as categorias oficiais do arXiv (`Ground Truth`).


## 1. Configuração do Ambiente e Carregamento dos Dados

Importamos as bibliotecas necessárias para processamento de linguagem natural, modelagem probabilística e visualizações interativas com **Plotly**.


In [1]:
import os
import re
import pandas as pd
import numpy as np

from sklearn.feature_extraction import text
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

# Configura o renderizador do Plotly para compatibilidade total com notebooks
pio.renderers.default = "notebook_connected"

# Configuração de exibição do Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

# Leitura dos resultados do LLDA
csv_path = "experimento_aapd_comparativo.csv"
if not os.path.exists(csv_path):
    csv_path = os.path.join(os.path.dirname(__file__), "experimento_aapd_comparativo.csv")

df = pd.read_csv(csv_path)

print(f"Dataset carregado com sucesso!")
print(f"Total de documentos: {len(df):,} | Tópicos únicos do LLDA: {df['topico_llda'].nunique():,}")
df[['doc_id', 'classes_reais_aapd', 'topico_llda', 'confianca_llda']].head(5)


Dataset carregado com sucesso!
Total de documentos: 2,000 | Tópicos únicos do LLDA: 1,989


,doc_id,classes_reais_aapd,topico_llda,confianca_llda
0,51,"math.CO, cs.DM",Graph Edge Rabbit Hunting Problem,0.333333
1,178,"physics.soc-ph, cs.DL",Scientific Research Collaboration Dynamics,0.333333
2,209,"cs.SI, physics.soc-ph",network seed identification measures,0.333333
3,228,"cs.IT, math.IT",Quantum code construction from cyclic codes,0.333333
4,285,"cs.DC, cs.DS",Sparse Matrix Storage Format Adaptation,0.333333


## 2. Pré-Processamento e Engenharia de Vocabulário

Para agregar os tópicos do LLDA de forma informativa, removemos:
1. Aspas, quebras de linha acidentais e pontuações remanescentes.
2. **Stopwords de Domínio Acadêmico:** Termos ubíquos na literatura científica (como *'algorithm'*, *'method'*, *'problem'*, *'approach'*, *'paper'*, *'analysis'*), que aparecem transversalmente em quase todos os resumos e obscureceriam os nichos temáticos reais.


In [2]:
def clean_topic_text(val):
    """Normaliza o texto do tópico para vetorização uniforme."""
    if not isinstance(val, str):
        return ""
    txt = val.replace('\n', ' ').strip('\"\'`')
    txt = re.sub(r'\s+', ' ', txt)
    return txt.strip()

df['topico_llda_clean'] = df['topico_llda'].apply(clean_topic_text)

# Stopwords em inglês enriquecidas com meta-termos acadêmicos genéricos
academic_meta_stops = {
    'algorithm', 'algorithms', 'method', 'methods', 'problem', 'problems',
    'approach', 'approaches', 'analysis', 'paper', 'study', 'technique',
    'techniques', 'based', 'using', 'proposed', 'new', 'framework', 'system', 'systems'
}
domain_stop_words = list(text.ENGLISH_STOP_WORDS.union(academic_meta_stops))

# Vetorizador configurado com unigramas e bigramas
vectorizer = CountVectorizer(
    stop_words=domain_stop_words,
    min_df=2,
    ngram_range=(1, 2),
    token_pattern=r'(?u)\b[a-zA-Z]{3,}\b'
)

dtm = vectorizer.fit_transform(df['topico_llda_clean'])
feature_names = vectorizer.get_feature_names_out()

print(f"Matriz Documento-Termo (DTM) gerada:")
print(f" - Documentos: {dtm.shape[0]:,}")
print(f" - Termos no Vocabulário (unigramas e bigramas): {dtm.shape[1]:,}")
print(f" - Densidade da matriz: {(dtm.nnz / (dtm.shape[0] * dtm.shape[1])) * 100:.2f}%")


Matriz Documento-Termo (DTM) gerada:
 - Documentos: 2,000
 - Termos no Vocabulário (unigramas e bigramas): 1,327
 - Densidade da matriz: 0.27%


## 3. Seleção do Número Ideal de Macro-Tópicos ($K$)

Testamos valores candidatos de $K \in [5, 8, 10, 12, 15, 20]$ calculando a **Perplexidade** do modelo LDA.  
Na modelagem de tópicos, uma perplexidade menor ou estabilizada indica melhor capacidade preditiva do modelo sobre as coocorrências lexicais.


In [3]:
k_candidates = [5, 8, 10, 12, 15, 20]
grid_results = []

for k in k_candidates:
    lda_k = LatentDirichletAllocation(
        n_components=k,
        random_state=42,
        max_iter=25,
        learning_method='batch',
        n_jobs=-1
    )
    lda_k.fit(dtm)
    perp = lda_k.perplexity(dtm)
    score = lda_k.score(dtm)
    grid_results.append({
        'K': k,
        'Perplexidade': round(perp, 2),
        'Log-Likelihood': round(score, 2)
    })

df_grid = pd.DataFrame(grid_results)
display(df_grid)


,K,Perplexidade,Log-Likelihood
0,5,1258.11,-52017.12
1,8,1394.07,-52765.01
2,10,1514.74,-53370.04
3,12,1537.91,-53480.64
4,15,1606.75,-53799.79
5,20,1757.50,-54453.35


### Visualização 1: Curva de Perplexidade por Número de Macro-Tópicos ($K$)


In [4]:
fig_perp = px.line(
    df_grid,
    x='K',
    y='Perplexidade',
    markers=True,
    title='Avaliação de Perplexidade do Modelo LDA para Agregação dos Tópicos LLDA',
    labels={'K': 'Número de Macro-Tópicos (K)', 'Perplexidade': 'Perplexidade no Corpus'}
)

fig_perp.update_traces(line_color='#1f77b4', marker=dict(size=9, color='#ff7f0e'))
fig_perp.update_layout(template='plotly_white', height=450)
fig_perp.show()


## 4. Ajuste do Modelo LDA de Agregação ($K=10$)

Selecionamos **$K=10$ macro-tópicos**, pois oferece um excelente equilíbrio entre parcimônia (poucos grupos para fácil consumo humano) e granularidade temática semântica.  
Abaixo, ajustamos o modelo e extraímos as palavras-chave mais descritivas de cada macro-tópico.


In [5]:
K_OPTIMAL = 10

lda_aggregator = LatentDirichletAllocation(
    n_components=K_OPTIMAL,
    random_state=42,
    max_iter=35,
    learning_method='batch',
    n_jobs=-1
)

doc_macro_dist = lda_aggregator.fit_transform(dtm)

# Extração dos top 6 termos por macro-tópico
macro_descriptions = {}
macro_records = []

for idx, component in enumerate(lda_aggregator.components_):
    macro_id = idx + 1
    top_indices = component.argsort()[:-7:-1]
    top_terms = [feature_names[i] for i in top_indices]
    desc_str = ", ".join(top_terms)
    macro_descriptions[macro_id] = desc_str
    
    macro_records.append({
        'Macro-Tópico ID': macro_id,
        'Rótulo Descritivo (Palavras-Chave)': desc_str
    })

df_macro_overview = pd.DataFrame(macro_records)
display(df_macro_overview)


,Macro-Tópico ID,Rótulo Descritivo (Palavras-Chave)
0,1,"complexity, codes, linear, random, parity, identification"
1,2,"channel, optimization, capacity, channel capacity, interference, recovery"
2,3,"network, social, optimization, model, dynamics, coding"
3,4,"language, resource, allocation, data, natural, model"
4,5,"control, design, models, channel, logic, optimization"
5,6,"matrix, machine, security, optimization, learning, model"
6,7,"error, data, information, power, graphs, retrieval"
7,8,"estimation, learning, channel, sparse, communication, coding"
8,9,"graph, quantum, theory, distributed, game, networks"
9,10,"detection, optimization, network, networks, community, management"


## 5. Distribuição dos Documentos entre os Macro-Tópicos

Atribuímos a cada um dos 2.000 documentos o seu macro-tópico predominante (aquele com maior probabilidade a posteriori) e a respectiva certeza da atribuição.


In [6]:
df['macro_topico_id'] = doc_macro_dist.argmax(axis=1) + 1
df['macro_probabilidade'] = doc_macro_dist.max(axis=1)
df['macro_topico_rotulo'] = df['macro_topico_id'].map(macro_descriptions)

# Estatísticas de distribuição de documentos
dist_macro = df['macro_topico_id'].value_counts().sort_index().reset_index()
dist_macro.columns = ['Macro-Tópico ID', 'Qtd_Documentos']
dist_macro['Percentual'] = (dist_macro['Qtd_Documentos'] / len(df) * 100).round(2)
dist_macro['Rótulo'] = dist_macro['Macro-Tópico ID'].map(macro_descriptions)

display(dist_macro)


,Macro-Tópico ID,Qtd_Documentos,Percentual,Rótulo
0,1,224,11.20,"complexity, codes, linear, random, parity, identification"
1,2,185,9.25,"channel, optimization, capacity, channel capacity, interference, recovery"
2,3,232,11.60,"network, social, optimization, model, dynamics, coding"
3,4,195,9.75,"language, resource, allocation, data, natural, model"
4,5,159,7.95,"control, design, models, channel, logic, optimization"
5,6,211,10.55,"matrix, machine, security, optimization, learning, model"
6,7,221,11.05,"error, data, information, power, graphs, retrieval"
7,8,189,9.45,"estimation, learning, channel, sparse, communication, coding"
8,9,188,9.40,"graph, quantum, theory, distributed, game, networks"
9,10,196,9.80,"detection, optimization, network, networks, community, management"


### Visualização 2: Contagem de Documentos por Macro-Tópico


In [7]:
fig_dist = px.bar(
    dist_macro,
    x='Qtd_Documentos',
    y='Rótulo',
    orientation='h',
    title='Volume de Artigos Agrupados por Macro-Tópico (Total: 2.000 Documentos)',
    text='Qtd_Documentos',
    color='Qtd_Documentos',
    color_continuous_scale='Blues'
)

fig_dist.update_layout(
    yaxis=dict(autorange='reversed'),
    xaxis_title='Número de Documentos Agrupados',
    yaxis_title='Macro-Tópico Agregador (LDA)',
    template='plotly_white',
    height=500
)

fig_dist.show()


### Visualização 3: Distribuição da Certeza da Agregação (Probabilidade a Posteriori Máxima)


In [8]:
fig_prob = px.box(
    df,
    x='macro_topico_id',
    y='macro_probabilidade',
    title='Certeza da Atribuição Documento-Macro-Tópico por Cluster',
    labels={'macro_topico_id': 'Macro-Tópico ID', 'macro_probabilidade': 'Probabilidade Posterior Máxima'},
    color='macro_topico_id'
)

fig_prob.update_layout(template='plotly_white', height=450, showlegend=False)
fig_prob.show()


## 6. Inspeção Qualitativa: Como os Micro-Tópicos do LLDA Foram Agrupados

Para avaliar a coerência semântica da agregação, inspecionamos abaixo amostras reais de micro-tópicos formulados pelo LLDA sob cada macro-tópico descoberto.


In [9]:
print("=" * 80)
print("EXEMPLOS DE MICRO-TÓPICOS LLDA DENTRO DE CADA MACRO-TÓPICO LDA:")
print("=" * 80)

for m_id in range(1, K_OPTIMAL + 1):
    sub = df[df['macro_topico_id'] == m_id].sort_values('macro_probabilidade', ascending=False)
    print(f"\n🔹 Macro-Tópico {m_id:02d}: {macro_descriptions[m_id]} ({len(sub)} docs)")
    for micro_top in sub['topico_llda_clean'].head(4):
        print(f"    • {micro_top}")


EXEMPLOS DE MICRO-TÓPICOS LLDA DENTRO DE CADA MACRO-TÓPICO LDA:

🔹 Macro-Tópico 01: complexity, codes, linear, random, parity, identification (224 docs)
    • Generalization of Binary Spatially Coupled Low Density Parity Check Codes
    • parity check matrix of convolutional code parity check matrix of qc code
    • Spatially Coupled Low Density Parity Check Codes
    • Binary N Code Parity Check Matrix Subset Size Redundancy Bound

🔹 Macro-Tópico 02: channel, optimization, capacity, channel capacity, interference, recovery (185 docs)
    • broadcast conflict resolution in multiple access channels dead sensor diagnosis wireless networks
    • Signal Signal Sets Modulation Channel Capacity Channel Capacity Region
    • Multiuser Gaussian Multiple Access Channel
    • Fading Gaussian Multiple Access Channel

🔹 Macro-Tópico 03: network, social, optimization, model, dynamics, coding (232 docs)
    • online learning literature deterministic pathwise statements martingales regret bounds tail

## 7. Correspondência Cruzada com as Categorias Oficiais do arXiv (Ground Truth)

Para verificar se a agregação do LDA respeita as grandes áreas científicas do arXiv, extraímos a classe primária de cada documento e cruzamos com os Macro-Tópicos descobertos.


In [10]:
df['classe_primaria'] = df['classes_reais_aapd'].apply(lambda x: str(x).split(',')[0].strip())

# Selecionamos as 8 classes primárias mais frequentes no corpus
top_classes = df['classe_primaria'].value_counts().head(8).index.tolist()
df_top_cls = df[df['classe_primaria'].isin(top_classes)]

ct = pd.crosstab(df_top_cls['macro_topico_id'], df_top_cls['classe_primaria'])
# Normaliza por linha para ver a composição percentual de cada macro-tópico
ct_pct = (ct.div(ct.sum(axis=1), axis=0) * 100).round(1)

display(ct_pct)


classe_primaria,cs.AI,cs.CV,cs.DS,cs.IT,cs.LG,cs.LO,math.CO,stat.ML
macro_topico_id,,,,,,,,
1,10.7,6.6,4.9,55.7,7.4,7.4,4.9,2.5
2,4.8,5.6,3.2,74.4,6.4,3.2,0.8,1.6
3,6.1,4.4,8.8,56.1,6.1,8.8,6.1,3.5
4,15.8,5.3,7.4,47.4,10.5,3.2,8.4,2.1
5,8.0,4.6,2.3,47.1,14.9,10.3,4.6,8.0
6,12.2,4.4,2.2,45.6,16.7,3.3,5.6,10.0
7,5.8,7.3,10.2,51.1,8.0,2.2,8.0,7.3
8,7.3,5.5,5.5,51.8,19.1,4.5,0.9,5.5
9,8.9,3.3,15.6,45.6,6.7,4.4,12.2,3.3


### Visualização 4: Heatmap de Correspondência (Macro-Tópicos LDA vs. Classes arXiv)


In [11]:
fig_heat = px.imshow(
    ct_pct.values,
    x=ct_pct.columns.tolist(),
    y=[f"Macro {i:02d}" for i in ct_pct.index],
    labels=dict(x="Categoria Principal do arXiv", y="Macro-Tópico LDA", color="Composição (%)"),
    title="Composição Relativa (%) das Categorias do arXiv em Cada Macro-Tópico LDA",
    color_continuous_scale='Viridis',
    text_auto=True
)

fig_heat.update_layout(template='plotly_white', height=520)
fig_heat.show()


## 8. Exportação do Dataset Enriquecido com Macro-Tópicos

Salvamos o DataFrame resultante com as novas colunas `macro_topico_id`, `macro_topico_rotulo` e `macro_probabilidade` para permitir futuras análises comparativas e agrupamentos adicionais.


In [ ]:
output_enriched_csv = "experimento_aapd_llda_macro_topicos.csv"
export_cols = [
    'doc_id',
    'classes_reais_aapd',
    'classe_primaria',
    'topico_llda',
    'confianca_llda',
    'macro_topico_id',
    'macro_topico_rotulo',
    'macro_probabilidade',
    'topico_lda_classico',
    'probabilidade_lda_classico',
    'resumo_texto'
]

df[export_cols].to_csv(output_enriched_csv, index=False)
print(f"Dataset enriquecido salvo com sucesso em: '{output_enriched_csv}'")
print(f"Total de linhas salvas: {len(df):,}")


## 9. Sumário Final e Conclusões

### Q&A
* **Por que aplicar LDA sobre os tópicos gerados pelo LLDA em vez de usar apenas um dos modelos?**
  A combinação cria uma abordagem hierárquica em dois estágios: o LLDA fornece formulações conceituais ricas, específicas e em linguagem natural (evitando a sobre-generalização inicial do LDA clássico), enquanto o LDA de segundo nível agrupa estatisticamente esses 1.989 micro-tópicos em 10 macro-tópicos interpretáveis e balanceados.
* **Quantos macro-tópicos foram selecionados e qual foi a distribuição de documentos?**
  Foram definidos **$K=10$ macro-tópicos**, obtendo uma distribuição bem equilibrada de artigos entre os clusters (variando entre 152 e 240 documentos por macro-tópico, com média de 200 artigos por cluster).

### Data Analysis Key Findings
* **Redução Dimensional Efetiva:** Os 1.989 micro-tópicos únicos gerados pelo LLDA foram condensados com sucesso em 10 macro-clusters temáticos, reduzindo a complexidade taxonômica em mais de 99,5% sem perder a rastreabilidade do micro-tópico original.
* **Coerência Temática Descoberta:** Os macro-tópicos capturaram domínios técnicos bem delineados da literatura científica do arXiv, incluindo:
  1. *Teoria da Informação e Comunicação Quântica* (`channel`, `capacity`, `quantum`, `codes`);
  2. *Redes Complexas e Redes Sociais* (`network`, `social`, `social network`, `dynamics`);
  3. *Otimização Convexa e Teoria dos Jogos* (`optimization`, `game`, `allocation`, `resource`);
  4. *Estruturas Algorítmicas e Grafos* (`graph`, `sparse`, `complexity`, `matrix`);
  5. *Aprendizado de Máquina e Modelagem Bayesiana* (`model`, `learning`, `bayesian`, `classification`).
* **Validação com Ground Truth do arXiv:** O heatmap de contingência demonstrou alta concentração esperada de classes como `cs.IT` em macro-tópicos de capacidade de canal, `cs.LG` / `stat.ML` em macro-tópicos de aprendizado de máquina, e `math.CO` / `cs.DS` em macro-tópicos de teoria de grafos e complexidade.

### Insights or Next Steps
* **Comparação com Agregação Densa (BERTopic):** Uma extensão direta e promissora é contrastar essa agregação probabilística do LDA com o agrupamento denso via embeddings de Sentence-Transformers + UMAP + HDBSCAN (conforme implementado no script `experimento_bertopic/bertopic_aggregator.py`).
* **Ajuste Fino de Rótulos por LLM:** É possível alimentar os 10 conjuntos de micro-tópicos de volta a um modelo de linguagem para gerar um nome macroscópico definitivo em alto nível para cada cluster.
